# Data Quality Validation Notebook
This notebook tests the validation functions implemented in `src/validation/quality_checks.py` against the bronze data files.

In [ ]:
import pandas as pd
import os
import sys

# Add the project root to sys.path to import from src
sys.path.append(os.path.abspath('..'))

from src.validation.quality_checks import (
    check_nulls, 
    check_duplicates, 
    check_ranges, 
    check_referential_integrity, 
    check_datatypes
)

: 

## 1. Load Data Files

In [ ]:
data_path = '../data/bronze/'
distributor_seasonality = pd.read_csv(os.path.join(data_path, 'distributor_seasonality_details.csv'))
holiday_list = pd.read_csv(os.path.join(data_path, 'holiday_list.csv'))
outlet_coordinates = pd.read_csv(os.path.join(data_path, 'outlet_coordinates.csv'))
outlet_master = pd.read_csv(os.path.join(data_path, 'outlet_master.csv'))
# transactions = pd.read_csv(os.path.join(data_path, 'transactions_history_final.csv')) # Large file, load carefully if needed

print("Data files loaded successfully.")

## 2. Test check_nulls
Checking nulls in `outlet_master`.

In [ ]:
cols_to_check = outlet_master.columns.tolist()
null_report = check_nulls(outlet_master, cols_to_check)
print("Null counts:", null_report)

## 3. Test check_duplicates
Checking for duplicate `outlet_id` in `outlet_master`.

In [ ]:
# Adjust 'outlet_id' to the actual column name if different
id_col = 'outlet_id' if 'outlet_id' in outlet_master.columns else outlet_master.columns[0]
dup_count = check_duplicates(outlet_master, [id_col])
print(f"Number of duplicates in {id_col}: {dup_count}")

## 4. Test check_ranges
Checking latitude and longitude ranges in `outlet_coordinates`.

In [ ]:
lat_col = 'latitude' if 'latitude' in outlet_coordinates.columns else 'lat'
lon_col = 'longitude' if 'longitude' in outlet_coordinates.columns else 'lon'

if lat_col in outlet_coordinates.columns:
    invalid_lat = check_ranges(outlet_coordinates, lat_col, -90, 90)
    print(f"Invalid Latitudes found: {len(invalid_lat)}")

if lon_col in outlet_coordinates.columns:
    invalid_lon = check_ranges(outlet_coordinates, lon_col, -180, 180)
    print(f"Invalid Longitudes found: {len(invalid_lon)}")

## 5. Test Referential Integrity
Checking if all outlets in `outlet_coordinates` exist in `outlet_master`.

In [ ]:
child_id = 'outlet_id' if 'outlet_id' in outlet_coordinates.columns else outlet_coordinates.columns[0]
parent_id = 'outlet_id' if 'outlet_id' in outlet_master.columns else outlet_master.columns[0]

missing_from_master = check_referential_integrity(outlet_coordinates, outlet_master, child_id, parent_id)
print(f"Outlets in coordinates missing from master: {len(missing_from_master)}")